In [1]:
import os
from pathlib import Path
import sys

import torch
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [3]:
from sentence_transformers import CrossEncoder

c:\Users\nizea\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [4]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv( override=True )

True

In [5]:
from simonbb.cis438.rag import load_documents, MultiQueryRerankRetriever, ParentDocumentRetriever, HypotheticalQuestionRetriever, build_rag

ModuleNotFoundError: No module named 'langgraph'

In [6]:
device = 'cpu'

embedding_model = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-large-en-v1.5",
    model_kwargs = {'device': device},
    encode_kwargs = {'normalize_embeddings': False}
)

reranker_model = CrossEncoder("BAAI/bge-reranker-large", device=device)

# Ingestion -> Retrieval -> Generation

## Step 1: Ingestion

Ingestion is the process of splitting documents into **chunks** which are then converted into **embedding** and stored in a **vector store**

In [29]:
docs, success_cnt, page_cnt = load_documents( Path.cwd().parent / 'syllabus.html' )

syllabus.html


In [30]:
print( docs[0].page_content[:200] ) # a document object contains 2 attributes: page_content and metadata

Course Description

This course introduces the basics of web-based agentic AI applications, with three main objectives. The primary objective is to introduce the design and implementation of agentic A


In [31]:
print(f"Files: {success_cnt}, Pages: {page_cnt}")

Files: 1, Pages: 1


In [32]:
text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", ". ", " ", ""], chunk_size=600, chunk_overlap=50)

In [33]:
chunks = text_splitter.split_documents(docs)

In [34]:
print(f"Chunks: {len(chunks)}")

Chunks: 15


In [35]:
print( chunks[0].metadata )

{'source': '/home/huaxia/Dropbox/is/ai/cis438/syllabus.html'}


In [36]:
vstore = Chroma.from_documents( documents=chunks, embedding=embedding_model )

## Step 2: Retrieval

In [37]:
retriever = vstore.as_retriever(search_kwargs={"k": 4})

In [38]:
query = 'who is the teaching assistant?'

In [39]:
docs = retriever.invoke( query )

In [40]:
def augment_context(input_dict):
    query = input_dict["query"]
    docs = input_dict["context"]
    snippets = []
    sources = {}
    for i, doc in enumerate(docs, start=1):
        sources[i] = {"text": doc.page_content, **doc.metadata}
        snippets.append(f"Snippet {i}: {doc.page_content}")
    return {"query": query, "context": "\n".join(snippets), "sources": sources}

In [41]:
context_aug = augment_context( {"query": query, "context": docs} )
context_aug

{'query': 'who is the teaching assistant?',
 'context': 'Snippet 1: Need help?\n\nTeaching Assistants\n\nJiawei Wang: jwang287@Simon.rochester.edu\n\nTA Office Hours: Friday: 1 -- 2 PM, at CSH 3-327 (aka Fishbowl), southeast end of the hallway with glass doors.\n\nFaculty Office Hours: Wednesday, 12 -- 1 PM, at CSH 3-310\n\nEmail: huaxia.rui@simon.rochester.edu\n\nPhone: 585-275-4278\nSnippet 2: In-Class Exercise: Travel Agent Dec 3 TH Agentic Design Patterns (II) Subgraph Supervisor Reflection LangSmith In-Class Exercise: Coursebot with Tools Dec 4 F Model Contex Protocol (MCP) Building MCP Server Consuming MCP Server Travel Agent App online by 1 PM Dec 8 T In-Person Presentation of Capstone Project Dec 10 (1:40 - 4:40) TH In-Person Presentation of Capstone Project\nSnippet 3: Quizzes 10% Quizzes are closed-book and are based only on my slides. I usually decide whether to give a quiz at the beginning of each class. Weblog Observatory App 10% Individual-based. Online by Nov 17 at 1 PM.

## Step 3: Generation

In [42]:
llm = ChatOpenAI( model='simonbb', max_tokens=4096, temperature=0.7, extra_body={"chat_template_kwargs": {"enable_thinking": False}},)

In [43]:
system_prompt = """You are a helpful assistant answering questions based on provided document snippets.

IMPORTANT INSTRUCTIONS:
1. Base your answer ONLY on the provided context snippets below.
2. Synthesize information from multiple snippets if needed.
3. If the snippets contain partial information, provide the best answer you can with what's available.
4. Only say "There wasn't enough information to answer the question" if the snippets truly contain NO relevant information.
5. Always cite your sources using the snippet numbers in parentheses, like (1), (2), etc.

The snippets are provided in this format:
Snippet 1: <text content>
Snippet 2: <text content>
...

{context}"""

In [44]:
prompt = ChatPromptTemplate.from_messages( [ ("system", system_prompt),  ("human", "{query}")  ])

In [45]:
messages = prompt.format_messages( context=context_aug["context"], query=query )

In [46]:
answer = llm.invoke(messages).content
answer

'The teaching assistant is Jiawei Wang, whose contact email is jwang287@Simon.rochester.edu (1).'

## Step 4: Format Output

In [77]:
def format_output(input_dict):
    answer = input_dict["answer"]
    sources = input_dict.get("sources", {})
    source_docs = []
    for k, v in sources.items():
        doc = Document(
            page_content=v.get('text', ''),
            metadata={key: val for key, val in v.items() if key != 'text'}
        )
        source_docs.append(doc)
    return {"result": answer, "source_docs": source_docs}

In [78]:
response = format_output( {"answer": answer, "sources": context_aug["sources"]} )

In [79]:
response["result"]

'The teaching assistant is Jiawei Wang, whose contact email is jwang287@Simon.rochester.edu (1).'

In [80]:
citations = response["source_docs"]

In [81]:
from IPython.display import Markdown, display

In [82]:
cites_md = ''
for i, citation in enumerate(citations, 1):
    source = citation.metadata.get('source', 'Unknown')
    page = citation.metadata.get('page', 'N/A')
    cites_md += f"{i}. **Chunk {i}** | Source: {source} | Page: {page} \n {citation.page_content}\n\n"

In [83]:
display(Markdown(cites_md))

1. **Chunk 1** | Source: /home/huaxia/Dropbox/is/ai/cis438/syllabus.html | Page: N/A 
 Need help?

Teaching Assistants

Jiawei Wang: jwang287@Simon.rochester.edu

TA Office Hours: Friday: 1 -- 2 PM, at CSH 3-327 (aka Fishbowl), southeast end of the hallway with glass doors.

Faculty Office Hours: Wednesday, 12 -- 1 PM, at CSH 3-310

Email: huaxia.rui@simon.rochester.edu

Phone: 585-275-4278

2. **Chunk 2** | Source: /home/huaxia/Dropbox/is/ai/cis438/syllabus.html | Page: N/A 
 In-Class Exercise: Travel Agent Dec 3 TH Agentic Design Patterns (II) Subgraph Supervisor Reflection LangSmith In-Class Exercise: Coursebot with Tools Dec 4 F Model Contex Protocol (MCP) Building MCP Server Consuming MCP Server Travel Agent App online by 1 PM Dec 8 T In-Person Presentation of Capstone Project Dec 10 (1:40 - 4:40) TH In-Person Presentation of Capstone Project

3. **Chunk 3** | Source: /home/huaxia/Dropbox/is/ai/cis438/syllabus.html | Page: N/A 
 Quizzes 10% Quizzes are closed-book and are based only on my slides. I usually decide whether to give a quiz at the beginning of each class. Weblog Observatory App 10% Individual-based. Online by Nov 17 at 1 PM. Travel Agent App 10% Individual-based. Online by Dec 4 at 1 PM. Capstone App 50% Individual-based, Online by Dec 8 at 1 PM.  In-person presentation on Dec 8 or Dec 10 Attendance 16% Your attendance score will not be affected unless you miss 8 or more classes. Beyond the exemption of 7 absences, each additional absence will reduce your attendance score by 2 points

4. **Chunk 4** | Source: /home/huaxia/Dropbox/is/ai/cis438/syllabus.html | Page: N/A 
 Recommended References

There is no textbook. But if you really need some references, here are two relatively recent ones, although they are both somewhat outdated by now.

Learning LangChain, by Mayo Oshin and Nuno Campos, 1st Edition, 2025

AI Agents and Applications, by Roberto Infante, 1st Edition, 2026

Hardware and Software



# Persistence

In [9]:
import chromadb
import sqlite3

In [10]:
store_path = Path.home() / 'chroma'
client = chromadb.PersistentClient(path=store_path)

In [11]:
conn = sqlite3.connect( store_path / "chroma.sqlite3" )
cursor = conn.cursor()
cursor.execute("SELECT collection, id FROM segments WHERE scope = 'VECTOR'")
folder_map = dict(cursor.fetchall())
conn.close()

In [62]:
for collection in client.list_collections():
    folder_name = folder_map.get( str(collection.id), "No folder created yet" )
    print( f"Name: {collection.name} | Count: {collection.count()} | Folder: {folder_name}" )

Name: cis438 | Count: 6 | Folder: 51775828-ee26-4ae2-9a76-3bc4e11a5e02
Name: paestum_parent | Count: 261 | Folder: 3762e976-cd22-4bb4-ba05-3b7e595cd2ce
Name: cis433 | Count: 9 | Folder: 63fab4c1-191c-47cc-a357-b10e6c5d8251
Name: gba411 | Count: 6 | Folder: f855e131-63e3-441b-9e00-dccaaa717308
Name: cis438_hypothetical_question | Count: 80 | Folder: c0200a0b-69ad-4fc4-8c47-8af7c39345b9
Name: cis438_original_chunks | Count: 20 | Folder: b3fec48c-8b9b-4cf1-8d0f-8cbf3a1e7f70
Name: paestum_child | Count: 512 | Folder: ac52f7b9-a403-4573-b149-baccabd499f4


In [67]:
from chromadb.errors import NotFoundError
def collection_exists(client, collection_name):
    try:
        client.get_collection(name=collection_name)
        return True
    except (ValueError, NotFoundError):
        return False

In [14]:
collection_name = 'cis438'
if not collection_exists(client, collection_name):
    Chroma.from_documents( documents=chunks, embedding=embedding_model, persist_directory=store_path, collection_name=collection_name )
else:
    print(f'collection {collection_name} exists')

collection cis438 exists


# Parent Document Retriever

In [16]:
kb_name = 'paestum'
kb_path = Path.home() / 'Data/text/Paestum'

In [17]:
retriever = ParentDocumentRetriever(
        persist_directory= store_path,
        embeddings = embedding_model,
        parent_collection = f'{kb_name}_parent',
        child_collection = f'{kb_name}_child',
    )

In [24]:
collection_name = f'{kb_name}_parent'
if not collection_exists(client, collection_name):
    print(f'collection {collection_name} does not exist. Ingesting ...')
    retriever.ingest( docs, source_dir=str(kb_path), parent_chunk_size=1000, child_chunk_size=500 )
else:
    print(f'collection {collection_name} exists')

collection paestum_parent exists


In [18]:
query = 'what is the most famous place to visit?'
retrieved_docs = retriever.invoke( query )

In [19]:
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nParent {i}:")
    print(doc.id)
    print(doc.metadata)
    print(doc.page_content)


Parent 1:
c10c4ed8-3ef9-4630-9783-09533404a2aa
{'moddate': '2018-01-12T14:48:00+01:00', 'creationdate': '2017-08-25T12:08:19+00:00', 'page': 13, 'aapl:keywords': '[]', 'page_label': '14', 'title': 'Microsoft Word - Färdig uppsats.docx', 'creator': 'Word', 'total_pages': 78, 'producer': 'Mac OS X 10.11.6 Quartz PDFContext', 'keywords': '', 'source': '/home/huaxia/Data/text/Paestum/PaestumRevisited-StocholmsUniversitet.pdf'}
and archaeologist Johann Joachim Winckelmann has been said to have written the first 
scientific work on Poseidonia-Paestum in his description of the ruins in Anmerkungen über die 
Baukunst den Alten, published in 1762.38 In 1768 Thomas Major of England published his 
volume The ruins of Paestum otherwise Poseidonia in Magna Graecia. In the very beginning 
of the book he writes ‘To the reader’: “The City of Paestum, or Poseidonia, whose Remains are 
here exhibited, hath been, ‘till very lately, almost buried in Oblivion.”39 Unsurprisingly it is the 
three Doric temp

In [21]:
llm = ChatOpenAI( model='simonbb', max_tokens=4096, temperature=0.7, extra_body={"chat_template_kwargs": {"enable_thinking": False}},)
rag_chain = build_rag( retriever, llm )

In [22]:
response = rag_chain.invoke( query )
response['answer']

'The most famous place to visit in Poseidonia-Paestum is the three Doric temples, which are highlighted in historical accounts and archaeological studies. These temples are considered among the most spectacular preserved echoes from a long-lost world (1). They attracted attention from early archaeologists like Johann Joachim Winckelmann and Thomas Major, who documented their significance in their works (1). The temples also hold cultural and historical value, as they are associated with the concept of place attachment and have inspired feelings of wonder and connection throughout history (5).'

# Hypothetical Question Retriever

In [57]:
kb_name = 'cis433'
docs, success_cnt, page_cnt = load_documents( Path.cwd().parent / 'cis433syllabus.html' )

cis433syllabus.html


In [69]:
collection_name = f'{kb_name}_hypothetical_question'

if not collection_exists(client, collection_name):
    print(f'collection {collection_name} does not exist. Ingesting ...')
    retriever = HypotheticalQuestionRetriever(
        llm = llm,
        persist_directory= store_path,
        embeddings = embedding_model,
        question_collection = f'{kb_name}_hypothetical_question',
        chunk_collection= f'{kb_name}_original_chunks',
        create_collection_if_not_exists = True
    )
    retriever.ingest( docs, source_dir=str(kb_path) )
else:
    print(f'collection {collection_name} exists')
    retriever = HypotheticalQuestionRetriever(
        llm = llm,
        persist_directory= store_path,
        embeddings = embedding_model,
        question_collection = f'{kb_name}_hypothetical_question',
        chunk_collection= f'{kb_name}_original_chunks'
    )

collection cis433_hypothetical_question exists


In [70]:
query = 'who is the ta?'
retrieved_docs = retriever.invoke( query )

In [71]:
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Chunk {i}:")
    print(doc.metadata)
    print(doc.page_content)

Chunk 1:
{'chunk_id': 'chunk-e3b62a4dcdef0230606bd3d5f19c833e', 'start_index': 7063, 'source_dir': '/home/huaxia/Data/text/Paestum', 'source': '/home/huaxia/Dropbox/is/ai/cis438/cis433syllabus.html'}
Need help?

Teaching Assistant(s)

Adam Li: sli73@simon.rochester.edu

Muyang Zhou: mzhou36@simon.rochester.edu

Faculty Office Hours: Wednesday, noon -- 1 PM, at Carol Simon Hall (CSH) 3-310

TA Office Hours: Friday: 2 -- 4 PM at CSH 3-327 (aka Fishbowl), a room at the end of the hallway with glass doors.

Email: huaxia.rui@simon.rochester.edu

Phone: 585-275-4278
Chunk 2:
{'start_index': 2461, 'chunk_id': 'chunk-45d37e2a86bf61944d4fda78d3075463', 'source': '/home/huaxia/Dropbox/is/ai/cis438/cis433syllabus.html', 'source_dir': '/home/huaxia/Data/text/Paestum'}
Learn instruction fine-tuning

Learn Quantization

Learn Low-Rank Adaptation (LoRA)

Learn fine-tuning Foundation Models

Learn the basics of Reinforcement Learning with Human Feedback (RLHF)

Relation with other Courses at Simon Bu

In [72]:
rag_chain = build_rag( retriever, llm )

In [73]:
response = rag_chain.invoke( query )
response['answer']

'The Teaching Assistants (TAs) for the course are Adam Li and Muyang Zhou. Their contact information is as follows:\n\n- Adam Li: sli73@simon.rochester.edu\n- Muyang Zhou: mzhou36@simon.rochester.edu\n\nThese details are provided in the first snippet (1).'